In [7]:
!nvidia-smi

%cd /content
!git clone https://github.com/nuhaminae/The-Conversion-Engine-Ground-Truth
%cd /content/The-Conversion-Engine-Ground-Truth

Sat May  2 13:32:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
%%capture
import os, re

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {
        "2.10": "0.0.34",
        "2.9": "0.0.33.post1",
        "2.8": "0.0.32.post2",
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install python-dotenv pyyaml

In [9]:
import os
from pathlib import Path
from getpass import getpass

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["WANDB_DISABLED"] = "true"

REPO = Path("/content/The-Conversion-Engine-Ground-Truth")
assert REPO.exists(), f"Repo not found: {REPO}"
os.chdir(REPO)
print("cwd:", os.getcwd())

hf_token = getpass("HF token, or press Enter to skip: ").strip()
os.environ["HF_TOKEN"] = hf_token
os.environ["WANDB_API_KEY"] = ""

!nvidia-smi

from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported

PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig

print("Unsloth + TRL imports OK")
print("bf16 supported:", is_bfloat16_supported())

cwd: /content/The-Conversion-Engine-Ground-Truth
HF token, or press Enter to skip: ··········
Sat May  2 13:33:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |   

In [10]:
import json
from pathlib import Path

train_file = Path("data/training_data/preferences_train.jsonl")
dev_file = Path("data/training_data/preferences_dev.jsonl")
config_file = Path("configs/training_config.yaml")
script_file = Path("src/training/train_judge.py")

for path in [train_file, dev_file, config_file, script_file]:
    assert path.exists(), f"Missing: {path}"

!ls -lah data/training_data

for path in [train_file, dev_file]:
    count = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            assert "prompt" in row and row["prompt"].strip()
            assert "chosen" in row and row["chosen"].strip()
            assert "rejected" in row and row["rejected"].strip()
            count += 1
    print(path, count, "valid rows")

!python -m py_compile src/training/train_judge.py
print("train_judge.py syntax OK")

total 204K
drwxr-xr-x 2 root root 4.0K May  2 13:32 .
drwxr-xr-x 5 root root 4.0K May  2 13:32 ..
-rw-r--r-- 1 root root  69K May  2 13:32 preferences_dev.jsonl
-rw-r--r-- 1 root root 121K May  2 13:32 preferences_train.jsonl
data/training_data/preferences_train.jsonl 65 valid rows
data/training_data/preferences_dev.jsonl 39 valid rows
train_judge.py syntax OK


In [11]:
!mkdir -p reports/training models/checkpoints models/judge

!PYTHONUNBUFFERED=1 python src/training/train_judge.py \
  --config configs/training_config.yaml \
  2>&1 | tee reports/training/training_run.log

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and 

In [12]:
print("Reports")
!find reports/training -maxdepth 2 -type f -print || true

print("\nModel files")
!find models -maxdepth 3 -type f | head -n 100 || true

print("\nTraining summary")
!cat reports/training/training_summary.json || true

!zip -r week11_judge_outputs.zip models reports configs/training_config.yaml
from google.colab import files

files.download("week11_judge_outputs.zip")

Reports
reports/training/dataset_summary.json
reports/training/training_summary.json
reports/training/training_config_used.yaml
reports/training/training_run.log

Model files
models/judge/chat_template.jinja
models/judge/README.md
models/judge/adapter_model.safetensors
models/judge/adapter_config.json
models/judge/special_tokens_map.json
models/judge/tokenizer_config.json
models/judge/tokenizer.json
models/model_card.md
models/checkpoints/checkpoint-17/chat_template.jinja
models/checkpoints/checkpoint-17/optimizer.pt
models/checkpoints/checkpoint-17/training_args.bin
models/checkpoints/checkpoint-17/rng_state.pth
models/checkpoints/checkpoint-17/trainer_state.json
models/checkpoints/checkpoint-17/README.md
models/checkpoints/checkpoint-17/adapter_model.safetensors
models/checkpoints/checkpoint-17/adapter_config.json
models/checkpoints/checkpoint-17/special_tokens_map.json
models/checkpoints/checkpoint-17/tokenizer_config.json
models/checkpoints/checkpoint-17/scheduler.pt
models/checkpo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>